In [11]:
import ee
from geemap import Map

try:
    ee.Initialize(project="ee708-rainfall-downscaling")
except:
    ee.Authenticate(auth_mode="localhost")
    ee.Initialize(project="ee708-rainfall-downscaling")

In [12]:
START_DATE ='1981-09-16'
END_DATE = '1981-09-17'

In [13]:
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY') \
    .filterDate(START_DATE, END_DATE) \
    .select('precipitation')

In [14]:
# 302x360 KM^2 area
# 0.03 degree = 1px change in image
uttarakhand_coords = [
    # Lon,  Lat
    [77.8, 31.9],
    [81.0, 31.9],
    [81.0, 28.75],
    [77.8, 28.75]
]

region = ee.Geometry.Polygon(uttarakhand_coords)

In [15]:
most_recent_chirps = chirps.sort("system:time_start", False).first()
precipitation_mask_chirps = most_recent_chirps.gt(0)
precipitation_mask_chrips = most_recent_chirps.updateMask(precipitation_mask_chirps)

In [16]:
center_lat = 30.32
center_lon = 78.88

map = Map(center=[center_lat, center_lon], zoom=7.4, basemap="HYBRID")
vis_params= {
    "min": 0.0,
    "max": 100.0,
    "palette": ['#ffffff', '#00ffff', '#0080ff', '#da00ff', '#ffa400', '#ff0000']
}
map

Map(center=[30.32, 78.88], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [17]:
map.addLayer(precipitation_mask_chrips.clip(region), vis_params, "CHIRPS")

In [18]:
NO_DATA_VALUE = 0

# Export the image, specifying the CRS, transform, and region.
dataset_list = chirps.toList(chirps.size())
n = ee.Number(chirps.size()).getInfo()
for i in range(n):
    image = ee.Image(dataset_list.get(i))
    projection = image.select("precipitation").projection().getInfo()
    date = image.get('system:time_start').getInfo()
    date_str = ee.Date(date).format('YYYY-MM-dd').getInfo()

    task = ee.batch.Export.image.toDrive(
        image=image,
        description=f'CHIRPS_daily_{date_str}',
        folder='CHIRPS_daily_Uttarakhand',
        crs=projection['crs'],
        crsTransform=projection['transform'],
        maxPixels=1e13,
        region=region,
        formatOptions={'noData': NO_DATA_VALUE},
    )

    task.start()

    print(f'Started export for {date_str}')

Started export for 1981-09-16
